# Fully online inference: LR55 + CAPNetLite (2018train20)

This notebook directly loads the trained LR55 and CAPNetLite weights and performs fully online inference from raw `.tim` IQ files. The route is:

`.tim -> BOI detection/frequency shift -> required FFT3 features -> LR55 -> (QPSK-like only) blind Rs/CFO -> simplified 11x15 CC -> CAPNetLite`.

It evaluates 2018 held-out batches 25-28 and the complete 2022 cross-domain set. No training or precomputed BOI/FFT/CC inference cache is used. All rows are checkpointed to one CSV: `fully_online_inference_results/LR55-CAPNetLite-2018train20.csv`.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

def _find_helper_project_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for root in [start, *start.parents]:
        if (root / "helper_function" / "__init__.py").exists():
            return root
    for root in [start, *start.parents]:
        candidate = root / "SPL_code" / "boi_shift_fft3_cfpattern_spl"
        if (candidate / "helper_function" / "__init__.py").exists():
            return candidate
    for root in [start, *start.parents][:3]:
        matches = list(root.glob("**/boi_shift_fft3_cfpattern_spl/helper_function/__init__.py"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError("Could not find helper_function from the current Jupyter working directory.")


PROJECT_DIR = _find_helper_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from helper_function.fully_online_pipeline_inference import (
    FullyOnlinePipeline,
    make_default_config,
)

SPLIT_MODE = "2018train20"
DATASET_ROOT = (PROJECT_DIR / "../../CSPB.ML_dataset").resolve()
MODEL_DIR = PROJECT_DIR / "fully_online_pipeline_models"
RESULT_DIR = PROJECT_DIR / "fully_online_inference_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = RESULT_DIR / "LR55-CAPNetLite-2018train20.csv"

CFG = make_default_config(DATASET_ROOT)
pipeline = FullyOnlinePipeline(
    dataset_root=DATASET_ROOT,
    model_dir=MODEL_DIR,
    split_mode=SPLIT_MODE,
    cfg=CFG,
    lr_vector_key="lr55",
    cap_model_key="capnet_lite",
    cc_algorithm="simplified_double",
)

# True resumes an interrupted inference run from the existing CSV.
RESUME_FROM_EXISTING_CSV = True

assert pipeline.lr_path.name == "lr55_2018train20.joblib"
assert pipeline.cap_path.name == "capnet_lite_c32_64_2018train20.pt"
assert len(pipeline.lr_feature_names) == 55

print("Dataset root:", DATASET_ROOT)
print("LR55 model:", pipeline.lr_path)
print("CAPNetLite model:", pipeline.cap_path)
print("Device:", pipeline.device)
print("LR feature count:", len(pipeline.lr_feature_names))
print("Output CSV:", RESULT_CSV)

Dataset root: /home/ruilin/SPL_Surreal/CSPB.ML_dataset
LR55 model: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/fully_online_pipeline_models/lr55_2018train20.joblib
CAPNetLite model: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/fully_online_pipeline_models/capnet_lite_c32_64_2018train20.pt
Device: cuda:0
LR feature count: 55
Output CSV: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/fully_online_inference_results/LR55-CAPNetLite-2018train20.csv


## Run inference and save the CSV

Negative-SNR 2018 rows retain their truth metadata but have empty prediction fields; their raw signal files are not loaded. The CSV schema is `dataset,index,snr_db,true_label,lr_prediction,nn_prediction`.

In [2]:
compact_results = pipeline.run_compact_csv(
    dataset_batches={
        2018: range(25, 29),
        2022: range(1, 29),
    },
    output_csv=RESULT_CSV,
    resume=RESUME_FROM_EXISTING_CSV,
    checkpoint_every=100,
)

print("Saved:", RESULT_CSV)
print("CSV columns:", compact_results.columns.tolist())
print("CSV rows:", len(compact_results))
display(compact_results.groupby("dataset", sort=True).size().rename("rows").to_frame())

2018train20: 2018:   0%|          | 0/16000 [00:00<?, ?signal/s]

2018train20: 2022:   0%|          | 0/112000 [00:00<?, ?signal/s]

Saved: /home/ruilin/SPL_Surreal/SPL_code/spl_sourcecode/fully_online_inference_results/LR55-CAPNetLite-2018train20.csv
CSV columns: ['dataset', 'index', 'snr_db', 'true_label', 'lr_prediction', 'nn_prediction']
CSV rows: 128000


,rows
dataset,
2018,16000
2022,112000


## Accuracy summary

Reconstruct the final LR55+CAPNetLite prediction from the saved CSV fields and report accuracy separately for each dataset. Rows without predictions (negative-SNR 2018 samples) are excluded.

In [3]:
import numpy as np
import pandas as pd

# Read the saved result so this cell also works after restarting the kernel.
accuracy_results = pd.read_csv(RESULT_CSV, keep_default_na=False)
accuracy_results["final_prediction"] = np.where(
    accuracy_results["lr_prediction"].eq("QPSK-like"),
    accuracy_results["nn_prediction"],
    accuracy_results["lr_prediction"],
)
evaluated = accuracy_results.loc[
    accuracy_results["final_prediction"].ne("")
].copy()
evaluated["correct"] = evaluated["final_prediction"].eq(
    evaluated["true_label"]
)

accuracy_summary = (
    evaluated.groupby("dataset", sort=True)
    .agg(
        evaluated_samples=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)
accuracy_summary["correct_predictions"] = accuracy_summary["correct_predictions"].astype(int)
accuracy_summary["accuracy_percent"] = 100.0 * accuracy_summary.pop("accuracy")
display(accuracy_summary.style.format({"accuracy_percent": "{:.4f}%"}))

for row in accuracy_summary.itertuples(index=False):
    print(
        f"{int(row.dataset)}: {int(row.correct_predictions):,}/"
        f"{int(row.evaluated_samples):,} correct, accuracy={row.accuracy_percent:.4f}%"
    )

,dataset,evaluated_samples,correct_predictions,accuracy_percent
0,2018,15960,14489,90.7832%
1,2022,112000,102495,91.5134%


2018: 14,489/15,960 correct, accuracy=90.7832%
2022: 102,495/112,000 correct, accuracy=91.5134%
